# 04 - LLM Review for Urgency Labels


In [13]:
from pathlib import Path
import json
import os
import re
import time

import pandas as pd

try:
    display
except NameError:
    def display(obj):
        print(obj)

## 1. Cau hinh


In [ ]:
_ENV_ROOT = Path.cwd()
if not (_ENV_ROOT / "data").exists():
    _ENV_ROOT = _ENV_ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(_ENV_ROOT / ".env")
except ImportError:
    pass

RUN_LLM_REVIEW = True
OPENAI_MODEL = "gpt-4.1-mini"
BATCH_SIZE = 10
MAX_REVIEW_ROWS = None
SLEEP_SECONDS = 0.2
SORT_HIGH_FIRST = True

## 2. Duong dan input/output

In [15]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_CSV = PROJECT_ROOT / "data" / "processed" / "student_voice_enriched.csv"
REVIEW_CANDIDATES_CSV = PROJECT_ROOT / "data" / "processed" / "urgency_review_candidates.csv"
LLM_REVIEW_CSV = PROJECT_ROOT / "data" / "processed" / "urgency_llm_review.csv"
OUTPUT_CSV = PROJECT_ROOT / "data" / "processed" / "student_voice_enriched_reviewed.csv"
REPORT_PATH = PROJECT_ROOT / "outputs" / "reports" / "llm_urgency_review_report.md"

REVIEW_CANDIDATES_CSV.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_CSV)
print("Review candidates:", REVIEW_CANDIDATES_CSV)
print("LLM review checkpoint:", LLM_REVIEW_CSV)
print("Output:", OUTPUT_CSV)
print("Report:", REPORT_PATH)

Input: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched.csv
Review candidates: d:\AI thuc chien\Student Voice Intelligence\data\processed\urgency_review_candidates.csv
LLM review checkpoint: d:\AI thuc chien\Student Voice Intelligence\data\processed\urgency_llm_review.csv
Output: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched_reviewed.csv
Report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\llm_urgency_review_report.md


## 3. Load enriched data

In [ ]:
df = pd.read_csv(INPUT_CSV)

REQUIRED_COLUMNS = [
    "text",
    "source_dataset",
    "split_original",
    "topic_std",
    "sentiment_std_3class",
    "urgency_level",
    "urgency_label_source",
    "urgency_rule_hits",
    "label_needs_review",
]

missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing:
    raise ValueError(f"Input thieu cot bat buoc: {missing}")

df["text"] = df["text"].fillna("").astype(str)
df["row_id"] = df.index

print("Shape:", df.shape)
display(df[["row_id", "text", "urgency_level", "urgency_label_source", "urgency_rule_hits"]].head())

Shape: (49141, 20)


,row_id,text,urgency_level,urgency_label_source,urgency_rule_hits
0,0,em xin chào các anh chị em em là sinh viên vừa...,low,default_low,NaN
1,1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,low,default_low,NaN
2,2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,low,default_low,NaN
3,3,chào thầy đăng ký học ghép . môn học hiện hệ t...,low,default_low,NaN
4,4,con bé vẫn hoang mang lắm .,low,default_low,NaN


## 4. Tao tap candidate can LLM review


In [17]:
candidate_mask = (
    (df["label_needs_review"] == 1)
    | (df["urgency_label_source"].isin(["rule_high", "rule_medium"]))
)

review_candidates = df.loc[candidate_mask, [
    "row_id",
    "text",
    "source_dataset",
    "split_original",
    "topic_std",
    "sentiment_std_3class",
    "urgency_level",
    "urgency_label_source",
    "urgency_rule_hits",
]].copy()

if SORT_HIGH_FIRST:
    priority = {"high": 0, "medium": 1, "low": 2}
    review_candidates["review_priority"] = review_candidates["urgency_level"].map(priority).fillna(9)
    review_candidates = review_candidates.sort_values(["review_priority", "row_id"]).drop(columns=["review_priority"])

review_candidates.to_csv(REVIEW_CANDIDATES_CSV, index=False, encoding="utf-8")

print("Review candidates:", len(review_candidates))
display(review_candidates["urgency_level"].value_counts().to_frame("count"))
display(review_candidates.head(10))

Review candidates: 921


,count
urgency_level,
medium,445
low,282
high,194


,row_id,text,source_dataset,split_original,topic_std,sentiment_std_3class,urgency_level,urgency_label_source,urgency_rule_hits
71,71,trình độ ông này thì tao không quen biết nên k...,NEU_ESC,train,personal_affairs,negative,high,rule_high,violence_harassment
126,126,bậy bạ quan chân quang trường đại học luật hà ...,NEU_ESC,train,social_affairs,negative,high,rule_high,safety
315,315,xin được phép chia buồn cùng gia đình 2 chị la...,NEU_ESC,train,personal_affairs,negative,high,rule_high,safety
600,600,khiếp đảm hẳn là đe dọa cơ mà .,NEU_ESC,train,others,negative,high,rule_high,violence_harassment
656,656,nghịch lý có cầu vượt đi bộ hs sinh viên vẫn b...,NEU_ESC,train,social_affairs,neutral,high,rule_high,safety
801,801,truong hợp lkn không van thrombectomy window ....,NEU_ESC,train,student_services,negative,high,rule_high,safety
1020,1020,tôn căng tin quái gì anh nên đổi tên thành tôn...,NEU_ESC,train,personal_affairs,negative,high,rule_high,violence_harassment
1310,1310,nguy hiểm bé k thuê chỗ trường chinh trường th...,NEU_ESC,train,student_services,positive,high,rule_high,safety
1456,1456,chào mọi người mình k61 và xin phép được xưng ...,NEU_ESC,train,student_services,negative,high,rule_high,violence_harassment
1500,1500,đăng ký hoãn thi em xin chào thầy cô và anh ch...,NEU_ESC,train,student_services,neutral,high,rule_high,safety


## 5. Guideline cho LLM


In [18]:
URGENCY_GUIDELINE = """
You are reviewing urgency labels for Vietnamese student feedback.

Return exactly one urgency label for each item: low, medium, or high.

Definitions:
- low: normal feedback, mild complaint, general dissatisfaction, praise, question, recommendation, or non-actionable discussion.
- medium: an issue that affects learning or student services but is not immediately dangerous. Examples: weak wifi, broken projector, broken classroom equipment, system error, registration issue, schedule/exam conflict, delayed response from office, overcrowded room.
- high: urgent safety or serious harm issue. Examples: fire, electrical leak/shock, accident, injury, violence, threat, harassment, sexual harassment, assault, serious danger, or the student truly cannot attend class/exam because of an incident.

Important negative examples:
- Do NOT label high for figurative or unrelated uses such as: com chay, chay hang, chay deadline, chay giao an, noi tieng, no so tai khoan, no luc.
- Toxic/profane language alone is not high unless it mentions violence, threat, harassment, danger, or a serious incident.
- Negative sentiment alone is not urgency.

Use the text content as the main evidence. The rule label is only a suggestion and may be wrong.
""".strip()

ALLOWED_URGENCY_LABELS = {"low", "medium", "high"}
ALLOWED_CONFIDENCE = {"low", "medium", "high"}

## 6. Ham tao prompt va parse JSON

In [19]:
def compact_text(text, max_chars=1200):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + " ..."


def build_batch_prompt(batch_df):
    items = []
    for row in batch_df.itertuples(index=False):
        items.append({
            "row_id": int(row.row_id),
            "text": compact_text(row.text),
            "rule_urgency_level": row.urgency_level,
            "rule_source": row.urgency_label_source,
            "rule_hits": "" if pd.isna(row.urgency_rule_hits) else str(row.urgency_rule_hits),
            "topic": row.topic_std,
            "sentiment": row.sentiment_std_3class,
        })

    return {
        "instruction": URGENCY_GUIDELINE,
        "output_format": {
            "reviews": [
                {
                    "row_id": "integer, must match input row_id",
                    "urgency_level_llm": "one of: low, medium, high",
                    "confidence": "one of: low, medium, high",
                    "reason": "short Vietnamese explanation, maximum 25 words"
                }
            ]
        },
        "items": items,
    }


def extract_json_object(text):
    text = str(text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback neu model boc JSON trong markdown fence hoac co text thua.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Khong tim thay JSON object trong response: {text[:300]}")
    return json.loads(match.group(0))


def validate_reviews(payload, expected_row_ids):
    if "reviews" not in payload or not isinstance(payload["reviews"], list):
        raise ValueError("Payload thieu list `reviews`.")

    rows = []
    seen = set()
    for item in payload["reviews"]:
        row_id = int(item["row_id"])
        label = str(item["urgency_level_llm"]).strip().lower()
        confidence = str(item.get("confidence", "medium")).strip().lower()
        reason = str(item.get("reason", "")).strip()

        if label not in ALLOWED_URGENCY_LABELS:
            raise ValueError(f"Label khong hop le cho row_id={row_id}: {label}")
        if confidence not in ALLOWED_CONFIDENCE:
            confidence = "medium"

        rows.append({
            "row_id": row_id,
            "urgency_level_llm": label,
            "urgency_llm_confidence": confidence,
            "urgency_llm_reason": reason,
        })
        seen.add(row_id)

    expected_row_ids = set(int(x) for x in expected_row_ids)
    missing_ids = expected_row_ids - seen
    if missing_ids:
        raise ValueError(f"Response thieu row_id: {sorted(missing_ids)}")

    return pd.DataFrame(rows)

## 7. LLM review function


In [20]:
def review_batch_with_openai(batch_df):
    from openai import OpenAI

    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError("Chua co OPENAI_API_KEY trong environment.")

    client = OpenAI()
    prompt_payload = build_batch_prompt(batch_df)

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {
                "role": "system",
                "content": "You are a careful data annotator. Return only valid JSON."
            },
            {
                "role": "user",
                "content": json.dumps(prompt_payload, ensure_ascii=False)
            },
        ],
        temperature=0,
    )

    payload = extract_json_object(response.output_text)
    return validate_reviews(payload, batch_df["row_id"].tolist())

## 8. Chay LLM review co checkpoint


In [21]:
if LLM_REVIEW_CSV.exists():
    llm_reviews = pd.read_csv(LLM_REVIEW_CSV)
else:
    llm_reviews = pd.DataFrame(columns=[
        "row_id",
        "urgency_level_llm",
        "urgency_llm_confidence",
        "urgency_llm_reason",
    ])

already_reviewed = set(llm_reviews["row_id"].astype(int).tolist()) if len(llm_reviews) else set()
pending = review_candidates[~review_candidates["row_id"].isin(already_reviewed)].copy()

if MAX_REVIEW_ROWS is not None:
    pending = pending.head(MAX_REVIEW_ROWS)

print("Already reviewed:", len(already_reviewed))
print("Pending this run:", len(pending))
print("RUN_LLM_REVIEW:", RUN_LLM_REVIEW)

if RUN_LLM_REVIEW and len(pending) > 0:
    new_reviews = []
    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending.iloc[start:start + BATCH_SIZE].copy()
        print(f"Reviewing batch {start // BATCH_SIZE + 1}: rows {batch['row_id'].tolist()}")
        batch_review = review_batch_with_openai(batch)
        new_reviews.append(batch_review)

        llm_reviews = pd.concat([llm_reviews, batch_review], ignore_index=True)
        llm_reviews = llm_reviews.drop_duplicates(subset=["row_id"], keep="last")
        llm_reviews.to_csv(LLM_REVIEW_CSV, index=False, encoding="utf-8")
        time.sleep(SLEEP_SECONDS)

    print("New reviews:", sum(len(x) for x in new_reviews))
else:
    llm_reviews.to_csv(LLM_REVIEW_CSV, index=False, encoding="utf-8")

print("Checkpoint rows:", len(llm_reviews))
display(llm_reviews.head())

Already reviewed: 30
Pending this run: 891
RUN_LLM_REVIEW: True
Reviewing batch 1: rows [4737, 4987, 5059, 5129, 5347, 5403, 5490, 5538, 5597, 5730]
Reviewing batch 2: rows [5891, 6043, 6068, 6159, 6695, 7428, 7544, 7747, 7911, 8034]
Reviewing batch 3: rows [8095, 8128, 8383, 8435, 8676, 8702, 8796, 8816, 8868, 9130]
Reviewing batch 4: rows [9290, 9408, 9422, 10329, 10380, 10564, 11119, 11823, 12135, 12305]
Reviewing batch 5: rows [12576, 12660, 12793, 12963, 13072, 13075, 13588, 14327, 14388, 14469]
Reviewing batch 6: rows [14768, 14834, 14959, 15234, 15236, 15354, 15375, 15406, 15599, 15944]
Reviewing batch 7: rows [16448, 16943, 17006, 17066, 17387, 17716, 17781, 17850, 17889, 17933]
Reviewing batch 8: rows [17984, 18335, 18380, 18627, 18654, 18893, 18990, 18993, 19071, 19138]
Reviewing batch 9: rows [19420, 19641, 19659, 19680, 19861, 19988, 20243, 20281, 20341, 20383]
Reviewing batch 10: rows [20578, 20655, 20697, 20710, 20751, 20807, 21032, 21082, 21355, 21395]
Reviewing batch 11

,row_id,urgency_level_llm,urgency_llm_confidence,urgency_llm_reason
0,71,medium,high,Nội dung có tố cáo gạ gẫm nhưng không rõ mức đ...
1,126,low,medium,"Nội dung dài về quyền và pháp luật, không đề c..."
2,315,high,high,"Nhắc đến vụ cháy gây thương vong, là sự cố an ..."
3,600,low,medium,"Câu nói chung chung, không rõ ràng về mối đe d..."
4,656,medium,high,Nêu vấn đề vi phạm giao thông tiềm ẩn nguy cơ ...


## 9. Merge LLM review vao dataset


In [22]:
llm_reviews = pd.read_csv(LLM_REVIEW_CSV) if LLM_REVIEW_CSV.exists() else pd.DataFrame()

review_cols = ["row_id", "urgency_level_llm", "urgency_llm_confidence", "urgency_llm_reason"]
for col in review_cols:
    if col not in llm_reviews.columns:
        llm_reviews[col] = pd.NA

llm_reviews = llm_reviews[review_cols].drop_duplicates(subset=["row_id"], keep="last")

reviewed_df = df.merge(llm_reviews, on="row_id", how="left")
reviewed_df["urgency_level_final"] = reviewed_df["urgency_level_llm"].fillna(reviewed_df["urgency_level"])
reviewed_df["urgency_final_source"] = reviewed_df["urgency_level_llm"].notna().map({True: "llm", False: "rule"})

display(reviewed_df["urgency_level"].value_counts().rename("rule_count").to_frame())
display(reviewed_df["urgency_level_final"].value_counts().rename("final_count").to_frame())
display(reviewed_df["urgency_final_source"].value_counts().rename("count").to_frame())

,rule_count
urgency_level,
low,48502
medium,445
high,194


,final_count
urgency_level_final,
low,48764
medium,335
high,42


,count
urgency_final_source,
rule,48220
llm,921


## 10. Kiem tra bat dong giua rule va LLM

In [ ]:
reviewed_only = reviewed_df[reviewed_df["urgency_level_llm"].notna()].copy()

if len(reviewed_only) > 0:
    reviewed_only["llm_disagrees_with_rule"] = reviewed_only["urgency_level_llm"] != reviewed_only["urgency_level"]
    print("Reviewed rows:", len(reviewed_only))
    print("Disagreements:", int(reviewed_only["llm_disagrees_with_rule"].sum()))
    display(pd.crosstab(reviewed_only["urgency_level"], reviewed_only["urgency_level_llm"], margins=True))
    display(reviewed_only[reviewed_only["llm_disagrees_with_rule"]][[
        "row_id", "text", "urgency_level", "urgency_level_llm", "urgency_llm_confidence", "urgency_llm_reason", "urgency_rule_hits"
    ]].head(30))

Reviewed rows: 921
Disagreements: 309


urgency_level_llm,high,low,medium,All
urgency_level,,,,
high,40,124,30,194
low,1,274,7,282
medium,1,146,298,445
All,42,544,335,921


,row_id,text,urgency_level,urgency_level_llm,urgency_llm_confidence,urgency_llm_reason,urgency_rule_hits
71,71,trình độ ông này thì tao không quen biết nên k...,high,medium,high,Nội dung có tố cáo gạ gẫm nhưng không rõ mức đ...,violence_harassment
126,126,bậy bạ quan chân quang trường đại học luật hà ...,high,low,medium,"Nội dung dài về quyền và pháp luật, không đề c...",safety
160,160,thông báo thông tin liên quan đến thi chứng ch...,medium,low,high,"Thông báo lịch thi và hướng dẫn, không phải vấ...",admin_issue
257,257,em xin hỏi một chút là lịch thi cuối kì của bọ...,medium,low,high,"Hỏi lịch thi và nghỉ, thuộc dạng câu hỏi thông...",admin_issue
569,569,ốm đi khám bệnh viện châm cứu trung ương phát ...,medium,high,high,"Bệnh nặng, cần hoãn thi, ảnh hưởng trực tiếp đ...",admin_issue
600,600,khiếp đảm hẳn là đe dọa cơ mà .,high,low,medium,"Câu nói chung chung, không rõ ràng về mối đe d...",violence_harassment
656,656,nghịch lý có cầu vượt đi bộ hs sinh viên vẫn b...,high,medium,high,Nêu vấn đề vi phạm giao thông tiềm ẩn nguy cơ ...,safety
667,667,đề nghị đại học luật hà nội dám công khai lịch...,medium,low,medium,Yêu cầu công khai lịch học không gây ảnh hưởng...,admin_issue
801,801,truong hợp lkn không van thrombectomy window ....,high,medium,high,Thông tin y tế liên quan đến bệnh viện nhưng k...,safety
1020,1020,tôn căng tin quái gì anh nên đổi tên thành tôn...,high,medium,high,Có nhắc đến quấy rối tình dục nhưng là cảnh bá...,violence_harassment


## 11. Export reviewed dataset va report

In [24]:
reviewed_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

def dist(series):
    counts = series.value_counts(dropna=False)
    return {key.item() if hasattr(key, "item") else key: int(value) for key, value in counts.items()}

num_llm_reviewed = int(reviewed_df["urgency_level_llm"].notna().sum())
num_disagree = 0
if num_llm_reviewed > 0:
    num_disagree = int((reviewed_df["urgency_level_llm"].notna() & (reviewed_df["urgency_level_llm"] != reviewed_df["urgency_level"])).sum())

report_lines = [
    "# LLM Urgency Review Report",
    "",
    "## Files",
    "",
    f"- Input: `{INPUT_CSV.relative_to(PROJECT_ROOT)}`",
    f"- Review candidates: `{REVIEW_CANDIDATES_CSV.relative_to(PROJECT_ROOT)}`",
    f"- LLM checkpoint: `{LLM_REVIEW_CSV.relative_to(PROJECT_ROOT)}`",
    f"- Output: `{OUTPUT_CSV.relative_to(PROJECT_ROOT)}`",
    "",
    "## Counts",
    "",
    f"- Review candidates: `{len(review_candidates)}`",
    f"- LLM reviewed rows: `{num_llm_reviewed}`",
    f"- LLM/rule disagreements: `{num_disagree}`",
    "",
    "## Distributions",
    "",
    f"- Rule urgency: `{dist(reviewed_df['urgency_level'])}`",
    f"- Final urgency: `{dist(reviewed_df['urgency_level_final'])}`",
    f"- Final source: `{dist(reviewed_df['urgency_final_source'])}`",
    "",
    "## Notes",
    "",
    "- `urgency_level` la nhan rule-based tu notebook 03.",
    "- `urgency_level_llm` la nhan LLM review neu da chay API.",
    "- `urgency_level_final` uu tien LLM label, fallback ve rule label neu chua review.",
    "- Nen human-check ngau nhien 100-200 mau da LLM review truoc khi train chinh thuc.",
]

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved reviewed dataset:", OUTPUT_CSV)
print("Saved report:", REPORT_PATH)

Saved reviewed dataset: d:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched_reviewed.csv
Saved report: d:\AI thuc chien\Student Voice Intelligence\outputs\reports\llm_urgency_review_report.md
